---
## 1. Environment Setup

In [18]:
# ======================================================================
# 1.1 — Imports
# ======================================================================
import os, shutil, json, yaml, random, datetime
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)
print('✅ Imports')

✅ Imports


In [19]:
# ======================================================================
# 1.2 — EFS Detection + Load Sprint 1 Path Config
# ======================================================================
# Sprint 1's data_exploration notebook (Notebook 00) saved a JSON file
# with the paths to raw labels, meta.yaml, and image directories.
# We reuse this so we don't need to re-discover paths.

_CANDIDATES = [
    Path("/home/sagemaker-user/user-default-efs"),
    Path("/home/sagemaker-user"),
    Path.home() / "user-default-efs",
    Path.home(),
]
EFS = next((c for c in _CANDIDATES if (c/"IronGear").exists()), _CANDIDATES[0])
IRONGEAR = EFS / "IronGear"
DATA_DIR = IRONGEAR / "data"

# Load paths saved by Sprint 1's data_exploration notebook
path_cfg_file = DATA_DIR / "reports" / "dataset_paths.json"
assert path_cfg_file.exists(), (
    f"dataset_paths.json not found at {path_cfg_file}\n"
    f"This file was created by Sprint 1's data_exploration notebook."
)
with open(path_cfg_file) as f:
    pc = json.load(f)

# Reconstruct Path objects from saved strings
DATASET_CSV = Path(pc["dataset_csv"])    # patient metadata CSV
META_YAML   = Path(pc["meta_yaml"])      # class name mapping YAML
LABELS_DIR  = Path(pc["labels_dir"])     # ORIGINAL raw labels (9 classes)
IMAGE_DIRS  = [Path(d) for d in pc["image_dirs"]]  # image_part* folders

# Output paths
YOLO_DIR    = DATA_DIR / "yolo_dataset"   # will be REBUILT for Sprint 2
REPORTS_DIR = DATA_DIR / "reports"
FIG_DIR     = DATA_DIR / "figures" / "processing_s2"
for d in [YOLO_DIR, REPORTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Sprint 2 output directory
SPRINT2_DIR = IRONGEAR / "Sprint2"
for sub in ['notebooks','models','figures','utils']:
    (SPRINT2_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"EFS:         {EFS}")
print(f"IronGear:    {IRONGEAR}")
print(f"Raw labels:  {LABELS_DIR}")
print(f"meta.yaml:   {META_YAML}")
print(f"YOLO output: {YOLO_DIR}")
print(f"\n✅ Raw labels dir exists: {LABELS_DIR.exists()}")
print(f"✅ meta.yaml exists:      {META_YAML.exists()}")

EFS:         /home/sagemaker-user/user-default-efs
IronGear:    /home/sagemaker-user/user-default-efs/IronGear
Raw labels:  /home/sagemaker-user/user-default-efs/IronGear/dataset/folder_structure/yolov5/labels
meta.yaml:   /home/sagemaker-user/user-default-efs/IronGear/dataset/folder_structure/yolov5/meta.yaml
YOLO output: /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset

✅ Raw labels dir exists: True
✅ meta.yaml exists:      True


---
## 2. Verify Raw Class IDs

**First action of Sprint 2.** Scan every `.txt` label file and count class IDs.

### Expected classes (GRAZPEDWRI-DX paper, Nagy et al. 2022):

| ID | Name | Expected Count |
|----|------|----------------|
| 0 | boneanomaly | ~274 |
| 1 | bonelesion | ~45 |
| 2 | foreignbody | ~38 |
| 3 | fracture | ~6,600 |
| 4 | metal | ~1,200 |
| 5 | periosteal | ~1,000 |
| 6 | pronatorsign | ~800 |
| 7 | softtissue | ~400 |
| 8 | text | ~23,722 |

> ⚠️ Do NOT assume — verify empirically.

In [20]:
# ======================================================================
# 2.1 — Load meta.yaml to get raw class names
# ======================================================================
with open(META_YAML) as f:
    meta = yaml.safe_load(f)

names = meta.get("names", {})
RAW_CLASS_MAP = (
    {i: n for i, n in enumerate(names)} if isinstance(names, list)
    else {int(k): v for k, v in names.items()}
)

print("Raw classes from meta.yaml:")
for rid, rname in sorted(RAW_CLASS_MAP.items()):
    print(f"  {rid}: {rname}")

Raw classes from meta.yaml:
  0: boneanomaly
  1: bonelesion
  2: foreignbody
  3: fracture
  4: metal
  5: periostealreaction
  6: pronatorsign
  7: softtissue
  8: text


In [21]:
# ======================================================================
# 2.2 — Scan raw label files and count annotations
# ======================================================================

def count_raw_labels(labels_dir):
    """Count annotations per class ID in the raw label files."""
    counts = Counter()
    n_files = 0
    for lf in tqdm(sorted(labels_dir.glob("*.txt")), desc="Scanning raw labels"):
        n_files += 1
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    counts[int(p[0])] += 1
    return counts, n_files

raw_counts, n_raw_files = count_raw_labels(LABELS_DIR)

print(f"\nScanned {n_raw_files:,} raw label files")
print(f"Total annotations: {sum(raw_counts.values()):,}\n")

print(f"{"ID":>4}  {"Raw Name":<25}  {"Count":>8}  {"% ":>6}")
print("-" * 50)
total = sum(raw_counts.values())
for rid in sorted(raw_counts.keys()):
    rname = RAW_CLASS_MAP.get(rid, '???')
    c = raw_counts[rid]
    print(f"{rid:>4}  {rname:<25}  {c:>8,}  {c/total*100:>5.1f}%")
print("-" * 50)

# Verify all expected IDs present
expected = set(RAW_CLASS_MAP.keys())
found = set(raw_counts.keys())
if found == expected:
    print(f"\n✅ All {len(found)} raw class IDs verified")
elif found < expected:
    print(f"\n⚠️ Missing IDs: {expected - found}")
else:
    print(f"\n❌ Unexpected IDs: {found - expected}")

Scanning raw labels:   0%|          | 0/20327 [00:00<?, ?it/s]


Scanned 20,327 raw label files
Total annotations: 47,443

  ID  Raw Name                      Count      % 
--------------------------------------------------
   0  boneanomaly                     276    0.6%
   1  bonelesion                       45    0.1%
   2  foreignbody                       8    0.0%
   3  fracture                     18,090   38.1%
   4  metal                           818    1.7%
   5  periostealreaction            3,453    7.3%
   6  pronatorsign                    567    1.2%
   7  softtissue                      464    1.0%
   8  text                         23,722   50.0%
--------------------------------------------------

✅ All 9 raw class IDs verified


---
## 3. Define Sprint 2 Class Remapping

| Original ID | Name | → | Project ID | Project Name | Action |
|---|---|---|---|---|---|
| 0 | boneanomaly | | — | — | **DROP** |
| 1 | bonelesion | | — | — | **DROP** |
| 2 | foreignbody | | — | — | **DROP** |
| 3 | fracture | | 0 | fracture | REMAP |
| 4 | metal | | 1 | metal_implant | REMAP |
| 5 | periosteal | | 2 | periosteal_reaction | REMAP |
| 6 | pronatorsign | | 3 | pronator_sign | REMAP |
| 7 | softtissue | | — | — | **DROP** |
| 8 | text | | 4 | text | REMAP |

In [22]:
# ======================================================================
# 3.1 — Sprint 2 target classes and remapping
# ======================================================================
# Maps EXACT lowercase raw names from meta.yaml to project names.
# IMPORTANT: keys must match meta.yaml exactly.

RAW_TO_PROJECT = {
    "fracture"          : "fracture",
    "metal"             : "metal_implant",
    "periostealreaction": "periosteal_reaction",
    "pronatorsign"      : "pronator_sign",
    "text"              : "text",               # NEW in Sprint 2
    # DROPPED from Sprint 1: softtissue (AP=0.307)
    # IGNORED (same as Sprint 1): boneanomaly, bonelesion, foreignbody
}

# Ordered list defines class IDs: index = class ID
PROJECT_CLASSES = [
    "fracture",            # 0
    "metal_implant",       # 1
    "periosteal_reaction", # 2
    "pronator_sign",       # 3
    "text",                # 4  (NEW — replaces soft_tissue)
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(PROJECT_CLASSES)}

# Build raw_id → project_id mapping
RAW_ID_TO_PROJ_ID = {}
print(f"{"Raw ID":>6}  {"Raw Name":<25}  {"\u2192  Project Name":<28}  Proj ID")
print("\u2500" * 72)
for raw_id, raw_name in sorted(RAW_CLASS_MAP.items()):
    proj_name = RAW_TO_PROJECT.get(raw_name.lower().strip())
    if proj_name is not None:
        proj_id = CLASS_TO_ID[proj_name]
        RAW_ID_TO_PROJ_ID[raw_id] = proj_id
        print(f"{raw_id:>6}  {raw_name:<25}  \u2192  {proj_name:<28}  {proj_id}")
    else:
        print(f"{raw_id:>6}  {raw_name:<25}  \u2192  {"(dropped)":<28}  \u2013")

assert len(RAW_ID_TO_PROJ_ID) == 5, (
    f"Expected 5 mapped classes, got {len(RAW_ID_TO_PROJ_ID)}.\n"
    f"Check RAW_TO_PROJECT keys match meta.yaml raw names above."
)
print(f"\n✅ All 5 target classes mapped")

# ---- Sprint 2 oversampling config ----
OVERSAMPLE = {
    "metal_implant"      : 5,
    "periosteal_reaction": 2,
    "pronator_sign"      : 5,
    # soft_tissue REMOVED (class dropped)
    # text: no oversampling (23,722 annotations)
}
print(f'Oversampling config: {OVERSAMPLE}')

Raw ID  Raw Name                   →  Project Name               Proj ID
────────────────────────────────────────────────────────────────────────
     0  boneanomaly                →  (dropped)                     –
     1  bonelesion                 →  (dropped)                     –
     2  foreignbody                →  (dropped)                     –
     3  fracture                   →  fracture                      0
     4  metal                      →  metal_implant                 1
     5  periostealreaction         →  periosteal_reaction           2
     6  pronatorsign               →  pronator_sign                 3
     7  softtissue                 →  (dropped)                     –
     8  text                       →  text                          4

✅ All 5 target classes mapped
Oversampling config: {'metal_implant': 5, 'periosteal_reaction': 2, 'pronator_sign': 5}


---
## 4. Rebuild YOLO Dataset with Sprint 2 Schema

Re-reads the **original raw labels**, applies Sprint 2's remapping,
and writes fresh label files to `data/yolo_dataset/`. Reuses the same
patient-level split from Sprint 1's `master_index.csv`.

In [23]:
# ======================================================================
# 4.1 — Load Sprint 1's master_index.csv (has patient split assignments)
# ======================================================================
master_csv = REPORTS_DIR / "master_index.csv"
assert master_csv.exists(), f'master_index.csv not found at {master_csv}'
master = pd.read_csv(master_csv)

assert "split" in master.columns, "master_index.csv has no split column!"
print(f"✅ Loaded master_index.csv: {len(master):,} rows")
print(f"   Splits: {master.split.value_counts().to_dict()}")

✅ Loaded master_index.csv: 20,327 rows
   Splits: {'train': 14243, 'test': 3115, 'val': 2969}


In [24]:
# ======================================================================
# 4.2 — YOLO label I/O helpers (same as Sprint 1)
# ======================================================================

def read_yolo(path):
    """Parse YOLO label file. Returns list of (class_id, cx, cy, w, h)."""
    rows = []
    try:
        with open(path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    rows.append((int(p[0]), float(p[1]), float(p[2]),
                                 float(p[3]), float(p[4])))
    except Exception:
        pass
    return rows

def write_yolo(rows, out_path):
    """Write (class_id, cx, cy, w, h) tuples to YOLO .txt file."""
    with open(out_path, "w") as f:
        for (c, cx, cy, w, h) in rows:
            f.write(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
print('✅ Helpers defined')

✅ Helpers defined


In [25]:
# ======================================================================
# 4.3 — Rebuild YOLO dataset from raw labels with Sprint 2 schema
# ======================================================================
# This REPLACES the Sprint 1 labels in yolo_dataset/.
# Images don't need re-copying (same images, same split).
# We only need to rewrite the label files with the new class IDs.
#
# Structure: yolo_dataset/images/{train,val,test}/
#            yolo_dataset/labels/{train,val,test}/

# Delete Sprint 1's build marker so we can rebuild
build_marker = YOLO_DIR / ".dataset_built"
os_marker = YOLO_DIR / ".oversampled"

# Create directories
for split in ['train', 'val', 'test']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# First: remove Sprint 1 oversampled copies (files with _os in stem)
removed = 0
for split in ['train', 'val', 'test']:
    for d in [YOLO_DIR/'images'/split, YOLO_DIR/'labels'/split]:
        for f in d.glob('*'):
            if '_os' in f.stem:
                f.unlink(); removed += 1
print(f'Removed {removed:,} Sprint 1 oversampled files')

# Counters
stats = {'copied': 0, 'negatives': 0, 'kept': 0, 'dropped': 0,
         'per_class': defaultdict(int)}
rare_images = defaultdict(list)  # for oversampling

print('\nRebuilding YOLO dataset with Sprint 2 schema...')
for _, row in tqdm(master.iterrows(), total=len(master), desc='Processing'):
    split = row['split']
    stem = row['stem']
    img_path = Path(row['image_path'])
    lbl_path = row['label_path']

    # ---- Remap raw labels with Sprint 2 schema ----
    remapped = []
    if row['has_label'] and pd.notna(lbl_path):
        for (raw_id, cx, cy, w, h) in read_yolo(lbl_path):
            proj_id = RAW_ID_TO_PROJ_ID.get(raw_id)
            if proj_id is not None:
                remapped.append((proj_id, cx, cy, w, h))
                stats['kept'] += 1
                stats['per_class'][PROJECT_CLASSES[proj_id]] += 1
            else:
                stats['dropped'] += 1

    if not remapped:
        stats['negatives'] += 1

    # ---- Write remapped label file ----
    write_yolo(remapped, YOLO_DIR / 'labels' / split / f'{stem}.txt')

    # ---- Copy image (skip if already exists from Sprint 1) ----
    dst_img = YOLO_DIR / 'images' / split / f"{stem}{row['ext']}"
    if not dst_img.exists() and img_path.exists():
        shutil.copy2(img_path, dst_img)
        stats['copied'] += 1

    # ---- Track rare train images for oversampling ----
    if split == 'train' and remapped:
        cls_in_img = {PROJECT_CLASSES[r[0]] for r in remapped}
        for cls_name in cls_in_img:
            if cls_name in OVERSAMPLE:
                rare_images[cls_name].append(
                    (img_path, lbl_path, remapped, row['ext'])
                )

print(f'\n✅ Dataset rebuilt')
print(f'  Images copied:    {stats["copied"]:,}')
print(f'  Annotations kept: {stats["kept"]:,}')
print(f'  Dropped (ignored): {stats["dropped"]:,}')
print(f'  Negative images:  {stats["negatives"]:,}')
print('\n  Per class:')
for cls in PROJECT_CLASSES:
    print(f'    {cls:<25}: {stats["per_class"][cls]:>7,}')

for split in ['train','val','test']:
    n = len(list((YOLO_DIR/'images'/split).glob('*')))
    print(f'  {split}: {n:,} images')

Removed 2,448 Sprint 1 oversampled files

Rebuilding YOLO dataset with Sprint 2 schema...


Processing:   0%|          | 0/20327 [00:00<?, ?it/s]


✅ Dataset rebuilt
  Images copied:    0
  Annotations kept: 46,650
  Dropped (ignored): 793
  Negative images:  15

  Per class:
    fracture                 :  18,090
    metal_implant            :     818
    periosteal_reaction      :   3,453
    pronator_sign            :     567
    text                     :  23,722
  train: 28,486 images
  val: 5,938 images
  test: 3,115 images


---
## 5. Apply Oversampling

In [26]:
# ======================================================================
# 5.2 — Apply oversampling to train split
# ======================================================================
os_added = defaultdict(int)

print('Oversampling rare classes in train split...')
for cls_name, n_copies in OVERSAMPLE.items():
    cls_id = CLASS_TO_ID[cls_name]
    for (img_path, lbl_path, remapped, ext) in rare_images.get(cls_name, []):
        stem = img_path.stem
        for copy_i in range(1, n_copies + 1):
            new_stem = f'{stem}_os{copy_i}'
            # Copy image
            src_img = YOLO_DIR / 'images' / 'train' / f'{stem}{ext}'
            dst_img = YOLO_DIR / 'images' / 'train' / f'{new_stem}{ext}'
            if src_img.exists() and not dst_img.exists():
                shutil.copy2(src_img, dst_img)
            # Write remapped label (already has Sprint 2 class IDs)
            dst_lbl = YOLO_DIR / 'labels' / 'train' / f'{new_stem}.txt'
            if not dst_lbl.exists():
                write_yolo(remapped, dst_lbl)
            os_added[cls_name] += 1

print('\nOversampling results:')
for cls, count in os_added.items():
    print(f'  {cls:<25}: +{count:,} copies')

# Final train count
total_train = len(list((YOLO_DIR/'images'/'train').glob('*')))
print(f'\nTrain total: {total_train:,} images')

Oversampling rare classes in train split...

Oversampling results:
  metal_implant            : +2,375 copies
  periosteal_reaction      : +3,090 copies
  pronator_sign            : +1,910 copies

Train total: 35,734 images


---
## 6. Verify Rebuilt Dataset

In [27]:
# ======================================================================
# 6.1 — Validate split counts and class IDs
# ======================================================================
print('Validating...')

for split in ['train', 'val', 'test']:
    img_dir = YOLO_DIR / 'images' / split
    lbl_dir = YOLO_DIR / 'labels' / split
    n_img = len(list(img_dir.glob('*')))
    n_lbl = len(list(lbl_dir.glob('*.txt')))

    # Count class IDs in this split's labels
    split_counts = Counter()
    for lf in lbl_dir.glob('*.txt'):
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    split_counts[int(p[0])] += 1

    print(f'\n{split}: {n_img:,} images, {n_lbl:,} labels')
    for cid in sorted(split_counts.keys()):
        cname = PROJECT_CLASSES[cid] if cid < len(PROJECT_CLASSES) else f'??? {cid}'
        print(f'    {cid}: {cname:<25} {split_counts[cid]:>7,}')

    # Check for unexpected IDs
    bad = {k for k in split_counts if k >= len(PROJECT_CLASSES)}
    if bad:
        print(f'    ❌ Unexpected class IDs: {bad}')
    else:
        print(f'    ✅ All IDs valid (0–{len(PROJECT_CLASSES)-1})')

Validating...

train: 35,734 images, 21,491 labels
    0: fracture                   21,952
    1: metal_implant               3,264
    2: periosteal_reaction         7,368
    3: pronator_sign               2,298
    4: text                       25,245
    ✅ All IDs valid (0–4)

val: 5,938 images, 2,969 labels
    0: fracture                    2,675
    1: metal_implant                 132
    2: periosteal_reaction           571
    3: pronator_sign                  87
    4: text                        3,456
    ✅ All IDs valid (0–4)

test: 3,115 images, 3,115 labels
    0: fracture                    2,752
    1: metal_implant                 142
    2: periosteal_reaction           518
    3: pronator_sign                  97
    4: text                        3,598
    ✅ All IDs valid (0–4)


---
## 7. Write `dataset.yaml`

In [28]:
# ======================================================================
# 7.1 — Write Sprint 2 dataset.yaml
# ======================================================================
# Note: path uses YOLO_DIR (absolute EFS path) to avoid symlink issues.
# On Kaggle, the training notebook writes its own yaml with Kaggle paths.

yaml_text = f"""# Iron Gear — Sprint 2: 5-class detection dataset
# Auto-generated by Sprint2/notebooks/data_processing.ipynb

path: {YOLO_DIR}
train: images/train
val:   images/val
test:  images/test

nc: 5

names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: text
"""

DATASET_YAML = YOLO_DIR / "dataset.yaml"
DATASET_YAML.write_text(yaml_text)
print(f"✅ dataset.yaml written to {DATASET_YAML}\n")
print(yaml_text)

✅ dataset.yaml written to /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/dataset.yaml

# Iron Gear — Sprint 2: 5-class detection dataset
# Auto-generated by Sprint2/notebooks/data_processing.ipynb

path: /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset
train: images/train
val:   images/val
test:  images/test

nc: 5

names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: text



---
## 8. Readiness Checklist

In [30]:
from pathlib import Path
from collections import defaultdict

val_dir = Path("/home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/images/val")
stems = defaultdict(list)
for f in val_dir.glob("*"):
    stems[f.stem].append(f)

dupes_removed = 0
for stem, files in stems.items():
    if len(files) > 1:
        # Keep the first, remove the rest
        for f in sorted(files)[1:]:
            f.unlink()
            dupes_removed += 1

print(f"Removed {dupes_removed} duplicates")
print(f"Val images now: {len(list(val_dir.glob('*')))}")

Removed 2969 duplicates
Val images now: 2969


In [31]:
# ======================================================================
# 8.1 — Final verification
# ======================================================================

# Count class IDs across all splits
all_ids = Counter()
for split in ['train','val','test']:
    for lf in (YOLO_DIR/'labels'/split).glob('*.txt'):
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p)==5: all_ids[int(p[0])] += 1

n_train = len(list((YOLO_DIR/'images'/'train').glob('*')))
n_val   = len(list((YOLO_DIR/'images'/'val').glob('*')))
n_test  = len(list((YOLO_DIR/'images'/'test').glob('*')))
n_os_test = len([p for p in (YOLO_DIR/'images'/'test').glob('*') if '_os' in p.stem])

checks = [
    ('Only IDs 0-4',        all(k in range(5) for k in all_ids)),
    ('5 classes present',    len(all_ids) == 5),
    (f'Train: {n_train:,}', n_train >= 15000),
    (f'Val: {n_val:,}',     2900 <= n_val <= 3200),
    (f'Test: {n_test:,}',   3000 <= n_test <= 3200),
    ('No os in test',       n_os_test == 0),
    ('dataset.yaml exists', DATASET_YAML.exists()),
]

print('='*50)
print('  SPRINT 2 DATA READINESS')
print('='*50)
ok = True
for desc, passed in checks:
    if not passed: ok = False
    print(f'  {"✅" if passed else "❌"}  {desc}')
print('='*50)

if ok:
    print('\n  🚀 READY! Upload data/yolo_dataset/ as Kaggle Dataset.')
    print('  Then run Sprint2/notebooks/02_train_evaluate.ipynb on Kaggle.')
else:
    print('\n  ⚠️ Fix issues above before proceeding.')

  SPRINT 2 DATA READINESS
  ✅  Only IDs 0-4
  ✅  5 classes present
  ✅  Train: 35,734
  ✅  Val: 2,969
  ✅  Test: 3,115
  ✅  No os in test
  ✅  dataset.yaml exists

  🚀 READY! Upload data/yolo_dataset/ as Kaggle Dataset.
  Then run Sprint2/notebooks/02_train_evaluate.ipynb on Kaggle.
